In [58]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [59]:
import tqdm
import sys
import os

In [60]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("all-MiniLM-L6-v2")

2026-08-19 09:58:35,802 - INFO - Use pytorch device_name: cpu
2026-08-19 09:58:35,802 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


2026-08-19 09:58:36,017 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-08-19 09:58:36,033 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
2026-08-19 09:58:36,167 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-08-19 09:58:36,184 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-08-19 09:58:36,314 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Red

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-08-19 09:58:37,431 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-19 09:58:37,448 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config.json "HTTP/1.1 200 OK"
2026-08-19 09:58:37,582 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-08-19 09:58:37,599 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transfor

In [61]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [62]:
sys.path.append(".")

In [63]:
from utils.datasets import SimpleSet, BerlinSparqlBenchmark
from utils.dbs.qlever import QleverDB
from utils.dbs.fuseki import FusekiDB
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import pandas as pd
import numpy as np
from utils.datasets.base_dataset import DataTensor

In [64]:
powers = np.arange(0, 6)  # extend on a more powerful machine
sizes = 10**powers

In [65]:
datasets: dict[int, BerlinSparqlBenchmark] = {}
raw_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Running BDSDM generation for size {size}...")
    dataset = BerlinSparqlBenchmark(base_dir=Path(f"./data/bsbm_{power}"), n=size)
    dataset.setup()
    datasets[power] = dataset
    # raw_sizes[power] = dataset.get_triple_count()

2026-08-19 09:58:38,665 - INFO - BSBM dataset already exists in data/bsbm_0, skipping generation
2026-08-19 09:58:38,665 - INFO - BSBM dataset already exists in data/bsbm_1, skipping generation
2026-08-19 09:58:38,666 - INFO - BSBM dataset already exists in data/bsbm_2, skipping generation
2026-08-19 09:58:38,666 - INFO - BSBM dataset already exists in data/bsbm_3, skipping generation
2026-08-19 09:58:38,666 - INFO - BSBM dataset already exists in data/bsbm_4, skipping generation
2026-08-19 09:58:38,667 - INFO - BSBM dataset already exists in data/bsbm_5, skipping generation


Running BDSDM generation for size 1...
Running BDSDM generation for size 10...
Running BDSDM generation for size 100...
Running BDSDM generation for size 1000...
Running BDSDM generation for size 10000...
Running BDSDM generation for size 100000...


In [66]:
encoded_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Encoding dataset of size {size}/power {power}...")
    dataset = datasets[power]
    encoded_sizes[power] = dataset.encode(encoding_model)
    #  dataset.get_triple_count(encoded=True)

Encoding dataset of size 1/power 0...
Encoded TTL file already exists at data/bsbm_0/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10/power 1...
Encoded TTL file already exists at data/bsbm_1/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100/power 2...
Encoded TTL file already exists at data/bsbm_2/dataset_encoded.nt, skipping encoding
Encoding dataset of size 1000/power 3...
Encoded TTL file already exists at data/bsbm_3/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10000/power 4...
Encoded TTL file already exists at data/bsbm_4/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100000/power 5...
Encoded TTL file already exists at data/bsbm_5/dataset_encoded.nt, skipping encoding


## BSBM queries

### Simple Use case: 
find 10 products with a specific encoded label


### Complex Use case: 
For a specific product find 10 other similar products via their product label. 


In [67]:
test_label = "house furniture storage container"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [68]:
test_tensor.to_literal().n3()

'"{\\"data\\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.07163400948047638, -0.04559392109513283, 0.044428009539842606, 0.0004388350062072277, -0.004044292028993368, 0.05173112079501152, 0.08436278998851776, -0.028667306527495384, -0.032494205981492996, 0.058949727565050125, -0.0051073553040623665, 0.09506019204854965, -0.029170090332627296, -0.07172352075576782, 0.036426812410354614, 0.01788988709449768, 0.07774496078491211, -0.01058284193277359, -0.02719942107796669, -0.014839782379567623, 0.004996407311409712, -0.05195301026105881, -0.01563340425491333, -0.0004883587826043367, 0.012572054751217365, -0.043

In [69]:
base_bsbm_set = datasets[3]
db_with_tensor_idx = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="QLever",
)
db_no_tensor_idx = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="QLever (no Tensor Vocabulary)",
    enable_tensor_index=False,
)
db_fuseki = FusekiDBNative(
    id="test_fuseki",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="Fuseki + RDFTensor",
)
possible_queries = db_with_tensor_idx.get_queries(test_tensor)

from utils.dbs.executable_db import ExecutableDB

dbs:list[ExecutableDB] = [db_with_tensor_idx, db_no_tensor_idx, db_fuseki]
print(possible_queries)


2026-08-19 09:58:39,624 - WARNING - Killing any existing process using port 26052 before starting the server
2026-08-19 09:58:39,637 - ERROR - Command failed with return code 1
2026-08-19 09:58:39,637 - INFO - Initialized QLeverDBNative with id=test, port_id=26052, dataset=bsbm, name=QLever, use_encoded_ttl=True, endpoint=http://localhost:26052/test-with-tidx/sparql
2026-08-19 09:58:39,638 - WARNING - Killing any existing process using port 26053 before starting the server
2026-08-19 09:58:39,649 - ERROR - Command failed with return code 1
2026-08-19 09:58:39,649 - INFO - Initialized QLeverDBNative with id=test, port_id=26053, dataset=bsbm, name=QLever (no Tensor Vocabulary), use_encoded_ttl=True, endpoint=http://localhost:26053/test-no-tidx/sparql
2026-08-19 09:58:39,651 - WARNING - Killing any existing process using port 29054 before starting the server


2026-08-19 09:58:39,661 - ERROR - Command failed with return code 1


{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>\nPREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nPREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>\nPREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>\nSELECT DISTINCT ?product  ?vector ?dist \nWHERE {\n?product rdf:label_embedding ?vector .\n?product bsbmv:productFeature ?feat .\nBIND(dtf:cosineSimilarity(?vector, "{\\"data\\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.0716340094

In [70]:
print(possible_queries[QUERY_DIFFICULTY.EASY][QUERY_TYPE.EMBEDDED])


PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>
PREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>
SELECT DISTINCT ?product  ?vector ?dist 
WHERE {
?product rdf:label_embedding ?vector .
?product bsbmv:productFeature ?feat .
BIND(dtf:cosineSimilarity(?vector, "{\"data\": [0.001923849806189537, 0.059069667011499405, -0.06259278953075409, -0.008788925595581532, 0.08017760515213013, 0.0004897266626358032, 0.049011558294296265, -0.016156358644366264, -0.04537816718220711, 0.011929630301892757, -0.003551364177837968, -0.0034783161245286465, 0.023264417424798012, 0.02449001744389534, -0.023595571517944336, -0.073823943734169, 0.014226873405277729, -0.00022305997845251113, -0.03111756220459938, 0.07163400948047638, -0.04559392109513283, 0.044428009539842606, 0.0004388350062072277, -0.0040

In [71]:
import os
os.getcwd()

'/nfsd/gracedata2/kantz/Dense-Vector-KG/benchmarks'

In [105]:
import time


with db_fuseki as db:
    timings = []
    for _ in tqdm.tqdm(range(10)):
        start_time = time.time()
        noised_tensor = test_tensor.to_numpy() + np.random.normal(scale=0.01, size=test_tensor.to_numpy().shape)
        noised_tensor = DataTensor.from_numpy(noised_tensor)

        results = db.query_auto(
            noised_tensor,
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=QUERY_TYPE.EMBEDDED,
        )
        elapsed_time = time.time() - start_time
        timings.append(elapsed_time*1e9)  # convert to nanoseconds
print("Median elapsed time:", np.median(timings))
results

2026-08-19 10:50:37,370 - WARNING - Killing any existing process using port 29054 before starting the server


2026-08-19 10:50:37,443 - ERROR - Command failed with return code 1
2026-08-19 10:50:37,443 - INFO - Loading dataset into Fuseki server from data/bsbm_3/dataset.nt
2026-08-19 10:50:37,444 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test_fuseki already exists, removing locks if any
2026-08-19 10:50:37,445 - INFO - Stopping server!
2026-08-19 10:50:37,508 - ERROR - Command failed with return code 1
2026-08-19 10:50:37,508 - INFO - Starting Fuseki server port 29054
2026-08-19 10:50:37,509 - ERROR - TENSOR_CP environment variable is not set
2026-08-19 10:50:37,510 - INFO - Using default TENSOR_CP=/nfsd/gracedata2/kantz/Dense-Vector-KG/jena-datatensor/jena-datatensor/target/*
2026-08-19 10:50:37,510 - ERROR - FUSEKI_HOME environment variable is not set
2026-08-19 10:50:37,511 - INFO - Using default FUSEKI_HOME=/home/kantz/apache-jena-fuseki-5.2.0
2026-08-19 10:50:37,511 - INFO - Using FUSEKI_HOME=/home/kantz/apache-jena-fuseki-5.2.0 and TENSOR_CP=/nfsd/gracedata2/kantz/Dense-Vector-KG/j

 382618Median elapsed time: 1823994874.9542236


29054/tcp:          


,product,vector,dist
0,bsbmi:dataFromProducer14/Product668,"{""data"": [0.06415609270334244, 0.0112423812970...",0.3215971863272915
1,bsbmi:dataFromProducer19/Product897,"{""data"": [0.02605944126844406, 0.0248944889754...",0.2904596318614194
2,bsbmi:dataFromProducer14/Product665,"{""data"": [-0.005199954379349947, 0.05737551674...",0.2699475543209706
3,bsbmi:dataFromProducer5/Product187,"{""data"": [0.08207504451274872, 0.0040897363796...",0.26294023744154954
4,bsbmi:dataFromProducer14/Product648,"{""data"": [-0.06801604479551315, 0.064525954425...",0.24247276754477667
5,bsbmi:dataFromProducer16/Product776,"{""data"": [-0.05298737809062004, 0.066510111093...",0.2314222704756462
6,bsbmi:dataFromProducer9/Product383,"{""data"": [-0.020090781152248383, -0.0054141189...",0.22833389495934472
7,bsbmi:dataFromProducer9/Product439,"{""data"": [-0.1157810166478157, 0.1029479503631...",0.222577668105088
8,bsbmi:dataFromProducer13/Product610,"{""data"": [-0.04075666889548302, 0.088839158415...",0.2200760708877457
9,bsbmi:dataFromProducer21/Product981,"{""data"": [0.07412167638540268, -0.010792593471...",0.21233305057905177


In [106]:
db_no_tensor_idx.server_log_file

PosixPath('scratch/bsbm/bsbm_3/db/test-no-tidx/test-no-tidx_run.log')

In [115]:
subtimings = db_fuseki.get_timings_per_query()
subtiming_records = []
for i, query_timings in enumerate(subtimings):
    for key, ts in query_timings.items():
        for t in ts:
            subtiming_records.append(
                {"query_id": i - 1, "timing_key": key, "timings": t}
            )
df_subtimings = pd.DataFrame(subtiming_records)
df_timings = pd.DataFrame(
    [
        {"query_id": i, "timing_key": "totalExecution", "timings": t}
        for i, t in enumerate(timings)
    ]
)
full_timings = pd.concat([df_subtimings, df_timings], ignore_index=True)


In [116]:
len(subtimings)

12

In [113]:
full_timings["timing_key"].describe()

count                 10
unique                 1
top       totalExecution
freq                  10
Name: timing_key, dtype: object

In [117]:
full_timings.groupby(["query_id", "timing_key"])[["timings"]].sum().reset_index().sort_values(by=["query_id", "timing_key"])

,query_id,timing_key,timings
0,0,totalExecution,4.219892e+09
1,1,tensorCosineSimilarity,1.641806e+09
2,1,tensorFromString,1.451797e+09
3,1,totalExecution,2.039872e+09
4,2,tensorCosineSimilarity,1.646995e+09
5,2,tensorFromString,1.270832e+06
6,2,totalExecution,1.575237e+09
7,3,tensorCosineSimilarity,1.113496e+09
8,3,tensorFromString,1.316819e+06
9,3,totalExecution,1.990099e+09


In [128]:
repetitions = 128


def run_queries(
    db: ExecutableDB, q_type: QUERY_TYPE, test_tensor: DataTensor, repetitions=128
):
    query_timings_ns = []
    available_qs = db.get_available_query_types()
    if q_type not in available_qs:
        print(f"Query type {q_type} not available for database {db.name}. Skipping.")
        return query_timings_ns
    g = tqdm.tqdm(range(repetitions), total=repetitions, unit="query", leave=False)
    g.set_description(f"Running queries on {db.name}/{q_type}: ")
    for _ in g:
        start_time = time.time()
        noised_tensor = test_tensor.to_numpy() + np.random.normal(
            scale=0.01, size=test_tensor.to_numpy().shape
        )
        noised_tensor = DataTensor.from_numpy(noised_tensor)

        results = db.query_auto(
            noised_tensor,
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=q_type,
        )
        elapsed_time = time.time() - start_time
        query_timings_ns.append(elapsed_time * 1e9)
    return query_timings_ns


all_timings = None
for q_type in [QUERY_TYPE.EMBEDDED, QUERY_TYPE.INDEX]:
    for db in dbs:
        db.clear_log()
        with db as db_instance:
            query_timings_ns = run_queries(
                db_instance, q_type, test_tensor, repetitions=8
            )
            if len(query_timings_ns) == 0:
                continue
            subtimings = db_instance.get_timings_per_query()
            subtiming_records = []
            for i, qt in enumerate(subtimings):
                for key, ts in qt.items():
                    for t in ts:
                        subtiming_records.append(
                            {
                                "query_id": i - 1,
                                "timing_key": key,
                                "timings": t,
                                "engine": db_instance.name,
                                "query_type": q_type.name.lower(),
                            }
                        )
            df_subtimings = pd.DataFrame(subtiming_records)
            df_timings = pd.DataFrame(
                [
                    {
                        "query_id": i,
                        "timing_key": "totalExecution",
                        "timings": t,
                        "engine": db_instance.name,
                        "query_type": q_type.name.lower(),
                    }
                    for i, t in enumerate(query_timings_ns)
                ]
            )
            full_timings = pd.concat([df_subtimings, df_timings], ignore_index=True)

            full_timings_sums = (
                full_timings.groupby(["query_id", "timing_key", "engine", "query_type"])[["timings"]]
                .count()
                .reset_index()
                .sort_values(by=["query_id", "timing_key"])
            )
            all_timings = (
                pd.concat([all_timings, full_timings_sums], ignore_index=True)
                if all_timings is not None
                else full_timings_sums
            )

2026-08-19 10:59:52,636 - INFO - Logging QLever setup to scratch/bsbm/bsbm_3/db/test-with-tidx/test-with-tidx_run.log
2026-08-19 10:59:52,637 - INFO - Loading dataset into QLever server from data/bsbm_3/dataset.nt
2026-08-19 10:59:52,637 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test-with-tidx already exists!
2026-08-19 10:59:52,638 - INFO - Stopping server!
2026-08-19 10:59:52,652 - ERROR - Command failed with return code 1
2026-08-19 10:59:52,653 - INFO - Starting QLever server on port 26052
2026-08-19 10:59:52,653 - INFO - Running command: qlever-server -i test-with-tidx --port 26052 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-08-19 10:59:52,655 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26052/test-with-tidx/sparql)


2026-08-19 10:59:53,666 - INFO - Server is up and responding to queries
2026-08-19 11:00:00,249 - INFO - Stopping server!                                                
2026-08-19 11:00:00,323 - ERROR - Command failed with return code 1
2026-08-19 11:00:00,324 - INFO - Logging QLever setup to scratch/bsbm/bsbm_3/db/test-no-tidx/test-no-tidx_run.log
2026-08-19 11:00:00,324 - INFO - Loading dataset into QLever server from data/bsbm_3/dataset.nt
2026-08-19 11:00:00,325 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test-no-tidx already exists!
2026-08-19 11:00:00,326 - INFO - Stopping server!
2026-08-19 11:00:00,338 - ERROR - Command failed with return code 1
2026-08-19 11:00:00,338 - INFO - Starting QLever server on port 26053
2026-08-19 11:00:00,339 - INFO - Running command: qlever-server -i test-no-tidx --port 26053 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-08-19 11:00:00,341 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (

 397096

2026-08-19 11:00:35,104 - INFO - Server is up and responding to queries
2026-08-19 11:00:35,480 - INFO - Stopping server!                                             
2026-08-19 11:00:35,548 - ERROR - Command failed with return code 1
2026-08-19 11:00:35,577 - INFO - Logging QLever setup to scratch/bsbm/bsbm_3/db/test-no-tidx/test-no-tidx_run.log
2026-08-19 11:00:35,577 - INFO - Loading dataset into QLever server from data/bsbm_3/dataset.nt
2026-08-19 11:00:35,578 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test-no-tidx already exists!
2026-08-19 11:00:35,578 - INFO - Stopping server!
2026-08-19 11:00:35,643 - ERROR - Command failed with return code 1
2026-08-19 11:00:35,644 - INFO - Starting QLever server on port 26053
2026-08-19 11:00:35,644 - INFO - Running command: qlever-server -i test-no-tidx --port 26053 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-08-19 11:00:35,646 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (htt

Query type QUERY_TYPE.INDEX not available for database Fuseki + RDFTensor. Skipping.
 398017

29054/tcp:          


In [125]:
from utils.format import map_df_readable

all_timings.to_csv("scratch/sub_bsbm_timings.csv", index=False)
all_timings = map_df_readable(all_timings)

In [138]:
all_timings["engine"].unique()

<ArrowStringArray>
['QLever', 'QLever (no Tensor Vocabulary)', 'Fuseki + RDFTensor']
Length: 3, dtype: str

In [144]:
all_timings[
    (all_timings["engine"] == "Fuseki + RDFTensor") & (all_timings["query_type"] == "embedded")
]

,query_id,timing_key,engine,query_type,timings
98,0,tensorCosineSimilarity,Fuseki + RDFTensor,embedded,20548
99,0,tensorFromString,Fuseki + RDFTensor,embedded,1000
100,0,totalExecution,Fuseki + RDFTensor,embedded,1
101,1,tensorCosineSimilarity,Fuseki + RDFTensor,embedded,20548
102,1,tensorFromString,Fuseki + RDFTensor,embedded,1
103,1,totalExecution,Fuseki + RDFTensor,embedded,1
104,2,tensorCosineSimilarity,Fuseki + RDFTensor,embedded,20548
105,2,tensorFromString,Fuseki + RDFTensor,embedded,1
106,2,totalExecution,Fuseki + RDFTensor,embedded,1
107,3,tensorCosineSimilarity,Fuseki + RDFTensor,embedded,20548


In [142]:
all_timings[
    (all_timings["engine"] == "QLever (no Tensor Vocabulary)") & (all_timings["query_type"] == "embedded")
]

,query_id,timing_key,engine,query_type,timings
57,-1,processQueryAndSendResult,QLever (no Tensor Vocabulary),embedded,1
58,0,processQueryAndSendResult,QLever (no Tensor Vocabulary),embedded,1
59,0,readWordFromDisk,QLever (no Tensor Vocabulary),embedded,20663
60,0,tensorCosineSimilarity,QLever (no Tensor Vocabulary),embedded,20548
61,0,tensorFromString,QLever (no Tensor Vocabulary),embedded,20551
62,0,totalExecution,QLever (no Tensor Vocabulary),embedded,1
63,1,processQueryAndSendResult,QLever (no Tensor Vocabulary),embedded,1
64,1,readWordFromDisk,QLever (no Tensor Vocabulary),embedded,20663
65,1,tensorCosineSimilarity,QLever (no Tensor Vocabulary),embedded,20548
66,1,tensorFromString,QLever (no Tensor Vocabulary),embedded,20551


In [126]:
all_timings["timing_key"].unique()


<ArrowStringArray>
['processQueryAndSendResult',            'readTensorData',
          'readWordFromDisk',    'tensorCosineSimilarity',
          'tensorFromBuffer',          'tensorFromString',
            'totalExecution',  'tensorIndexComputeResult']
Length: 8, dtype: str

In [127]:
timing_mapping = {
    "totalExecution": "Total (ms)",
    "readTensorData": "Read Tensors (ms)",
    "readWordFromDisk": "Read RDF (ms)",
    "tensorCosineSimilarity": "Cosine Similarity (ms)",
    "tensorFromBuffer": "Parse from Buffer (ms)",
    "tensorFromString": "Parse from String (ms)",
    "tensorIndexComputeResult": "Index Lookup (ms)",
}
all_timings["timing_key"] = all_timings["timing_key"].map(timing_mapping).fillna(all_timings["timing_key"])

# timing mappings to columns

all_timings_grouped = (
    all_timings.groupby(["engine", "query_type", "timing_key"])[["timings"]]
    .median()
    .reset_index()
    .sort_values(by=["engine", "query_type", "timing_key"])
)
all_timings_grouped["timings"] = all_timings_grouped["timings"] / 1e6  # convert to ms

all_timings_grouped


,engine,query_type,timing_key,timings
0,RDFTensor,Embedded,Cosine Similarity (ms),1431.662300
1,RDFTensor,Embedded,Parse from String (ms),0.953318
2,RDFTensor,Embedded,Total (ms),1757.084727
3,\systemname-Base,Embedded,Cosine Similarity (ms),9.089006
4,\systemname-Base,Embedded,Parse from String (ms),20.230249
5,\systemname-Base,Embedded,Read RDF (ms),29.624893
6,\systemname-Base,Embedded,Total (ms),1179.110169
7,\systemname-Base,Embedded,processQueryAndSendResult,1151.317713
8,\systemname-Base,Index,Index Lookup (ms),3.954499
9,\systemname-Base,Index,Parse from String (ms),0.002724


In [132]:
all_timings_pivot = all_timings_grouped.pivot_table(
    index=["engine", "query_type"], columns="timing_key", values="timings"
).reset_index().fillna("-").set_index(["engine", "query_type"])
all_timings_pivot

timing_key                  Cosine Similarity (ms) Index Lookup (ms)  \
engine           query_type                                            
RDFTensor        Embedded                1431.6623                 -   
\systemname-Base Embedded                 9.089006                 -   
                 Index                           -          3.954499   
\systemname-TV   Embedded                 8.300608                 -   
                 Index                           -          4.785185   

timing_key                  Parse from Buffer (ms)  Parse from String (ms)  \
engine           query_type                                                  
RDFTensor        Embedded                        -                0.953318   
\systemname-Base Embedded                        -               20.230249   
                 Index                           -                0.002724   
\systemname-TV   Embedded                 2.747562                0.020866   
                 Index                    0.113561                0.002975   

timing_key                  Read RDF (ms) Read Tensors (ms)   Total (ms)  \
engine           query_type                                                
RDFTensor        Embedded               -                 -  1757.084727   
\systemname-Base Embedded       29.624893                 -  1179.110169   
                 Index           0.193411                 -    25.285363   
\systemname-TV   Embedded        0.209392        188.608091   538.276553   
                 Index           0.461216          9.508288    25.979280   

timing_key                  processQueryAndSendResult  
engine           query_type                            
RDFTensor        Embedded                           -  
\systemname-Base Embedded                 1151.317713  
                 Index                       4.728329  
\systemname-TV   Embedded                  517.552285  
                 Index                       5.435574

In [ ]:
counts=[]
for power, size in zip(powers, sizes):
    difficulties = [
        QUERY_DIFFICULTY.HARD,
        QUERY_DIFFICULTY.EASY,
    ]
    dataset = datasets[power]
    db = QleverDBNative(
        id="timing-qlever",
        base_dir=Path(f"./scratch/bsbm/{dataset.base_dir.name}"),
        dataset=dataset,
        use_encoded_ttl=True,
    )
    with db:
        full_number_of_tensors = db.query("""
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
SELECT (COUNT(?v) AS ?count) WHERE {
    {
    SELECT ?s ?v WHERE {          
    ?s rdf:label_embedding ?v .
    }
    } UNION {
            SELECT ?s ?v WHERE {
        ?s rdf:comment_embedding ?v .}
    }
}
""")["count"].values[0]
        full_size = db.get_triple_count()
        counts.append({
            "power": power,
            "size": size,
            "full_size": full_size,
            "full_number_of_tensors": full_number_of_tensors,
        })
counts_df = pd.DataFrame(counts)
counts_df

2026-04-17 17:08:59,387 - WARNING - Killing any existing process using port 26087 before starting the server
2026-04-17 17:08:59,409 - ERROR - Command failed with return code 1
2026-04-17 17:08:59,410 - INFO - Initialized QLeverDBNative with id=timing-qlever, port_id=26087, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26087/timing-qlever-with-tidx/sparql
2026-04-17 17:08:59,410 - INFO - Logging QLever setup to scratch/bsbm/bsbm_0/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-04-17 17:08:59,411 - INFO - Loading dataset into QLever server from data/bsbm_0/dataset.nt
2026-04-17 17:08:59,412 - WARNING - DB directory scratch/bsbm/bsbm_0/db/timing-qlever-with-tidx already exists!
2026-04-17 17:08:59,412 - INFO - Stopping server!
2026-04-17 17:08:59,430 - ERROR - Command failed with return code 1
2026-04-17 17:08:59,430 - INFO - Starting QLever server on port 26087
2026-04-17 17:08:59,431 - INFO - Running command: qlever-server

,power,size,full_size,full_number_of_tensors
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792


In [ ]:
counts_df = counts_df.astype(int)
counts_df

,power,size,full_size,full_number_of_tensors
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792


In [ ]:
out_dir = Path("./scratch/results")
out_dir.mkdir(parents=True, exist_ok=True)
from utils.helpers import pretty_print_counts
pretty_print_counts(counts_df, out_dir / "bsbm_counts.tex")

,Power,Generation $t$,$n$,$n_{tensors}$
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792
